In [1]:


from sentence_transformers import SentenceTransformer
from PIL import Image
import requests

# Load CLIP model
clip_model = SentenceTransformer('clip-ViT-B-32')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sentence-transformers--clip-ViT-B-32/snapshots/327ab6726d33c0e22f920c83f2ff9e4bd38ca37f/0_CLIPModel
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [6]:
!pip install Pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 736.8/736.8 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0


In [2]:
import getpass
import os
import pinecone
PINECONE_API_KEY = getpass.getpass("Enter your Pinecone API key: ")

# Set environment variable and verify
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
assert PINECONE_API_KEY, "Please set your Pinecone API key"

# Initialize Pinecone client
pc = pinecone.Pinecone(api_key=PINECONE_API_KEY)

Enter your Pinecone API key: ··········


In [4]:
from pinecone import Pinecone, ServerlessSpec
img1 = Image.open("/content/cat.png")
img2 = Image.open("/content/dog.png")

# Encode images (assuming clip_model is already loaded)
image_embeddings = clip_model.encode([img1, img2]).tolist()
texts = ["dog photo", "cat photo"]

# Create Pinecone index for images if it doesn't exist
index_name = "images-index"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=512,   # depends on your model
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f" Created new index: {index_name}")
else:
    print(f" Using existing index: {index_name}")

# Connect to the index
img_index = pc.Index(index_name)

# Upsert image embeddings
img_index.upsert(vectors=[
    ("img1", image_embeddings[0], {"label": texts[0]}),
    ("img2", image_embeddings[1], {"label": texts[1]})
])

print(" Image embeddings uploaded to Pinecone!")

 Created new index: images-index
 Image embeddings uploaded to Pinecone!


In [5]:

query = "a cute cat"
query_vec = clip_model.encode([query]).tolist()
results = img_index.query(vector=query_vec[0], top_k=2, include_metadata=True)
print(results)


QueryResponse(matches=[{'id': 'img1',
 'metadata': {'label': 'dog photo'},
 'score': 0.286019295,
 'values': []}, {'id': 'img2',
 'metadata': {'label': 'cat photo'},
 'score': 0.203929067,
 'values': []}], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Wed, 18 Feb 2026 05:12:05 GMT', 'content-type': 'application/json', 'content-length': '223', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '32', 'x-envoy-upstream-service-time': '33', 'x-pinecone-response-duration-ms': '35', 'grpc-status': '0', 'server': 'envoy'}})


In [6]:
!pip install -q transformers torch

from transformers import AutoTokenizer, AutoModel
import torch
from pinecone import Pinecone, ServerlessSpec

#  Initialize Pinecone client
API_KEY = ""
pc = Pinecone(api_key=API_KEY)

#  Load BioBERT model
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")
model_bio = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

#  Function to generate embeddings
def embed_biomedical(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model_bio(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.tolist()

#  Example biomedical data
bio_texts = [
    "The protein p53 regulates the cell cycle.",
    "COVID-19 is caused by the SARS-CoV-2 virus."
]
bio_vectors = embed_biomedical(bio_texts)

#  Create or connect to a Pinecone index
index_name = "biomed-index"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=768,       # BioBERT embedding size
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f" Created new index: {index_name}")
else:
    print(f" Using existing index: {index_name}")

bio_index = pc.Index(index_name)


bio_index.upsert([
    (f"bio_{i}", bio_vectors[i], {"text": bio_texts[i]})
    for i in range(len(bio_texts))
])

query = "cell cycle regulation"
query_vec = embed_biomedical([query])[0]

results = bio_index.query(vector=query_vec, top_k=2, include_metadata=True)


print(" Biomedical Similarity Search Results:")
for match in results.matches:
    print(f"- {match.metadata['text']} (Score: {match.score:.3f})")

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Created new index: biomed-index
 Biomedical Similarity Search Results:
- COVID-19 is caused by the SARS-CoV-2 virus. (Score: 0.726)
- The protein p53 regulates the cell cycle. (Score: 0.837)
